In [ ]:
from pathlib import Path
import pandas as pd

# Define path directories
source_dir = Path(
    "yoursourcefolder"  # Change this to your source folder
)
output_dir = Path(
    "yourdirecttooutputfolder"  # Change this to your desired output folder
)

# Recursively locate all CSV files across all language folders
csv_files = list(source_dir.glob("**/*.csv"))
print(
    f"Found {len(csv_files)} CSV files. Deduplicating strictly by 'repo_name'...\n"
)

total_initial_rows = 0
total_cleaned_rows = 0

for file_path in csv_files:
    # Preserve relative subfolder structure (e.g., java/file.csv -> output_dir/java/file.csv)
    relative_path = file_path.relative_to(source_dir)
    target_path = output_dir / relative_path
    target_path.parent.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(file_path)
    initial_count = len(df)

    if "repo_name" in df.columns:
        # Keep only the first occurrence of each unique repository name
        df_clean = df.drop_duplicates(subset=["repo_name"], keep="first")
    else:
        # Fallback to column index 0 if column header naming differs slightly
        df_clean = df.drop_duplicates(subset=[df.columns[0]], keep="first")

    clean_count = len(df_clean)
    df_clean.to_csv(target_path, index=False)

    dropped = initial_count - clean_count
    total_initial_rows += initial_count
    total_cleaned_rows += clean_count

    print(f"[{relative_path}] {initial_count} -> {clean_count} (-{dropped})")

print("\n" + "=" * 50)
print("DEDUPLICATION SUMMARY")
print(f"Total Initial Rows  : {total_initial_rows:,}")
print(f"Unique Repos Saved  : {total_cleaned_rows:,}")
print(f"Duplicates Dropped  : {total_initial_rows - total_cleaned_rows:,}")
print(f"Cleaned dataset location:\n{output_dir}")